# Spotify usage analysis

This notebook will show stats based on Spotify historical audio stream usage. More information on this data, including how to obtain a personal copy, can be found in [Spotify's official support page](https://support.spotify.com/uk/article/data-rights-and-privacy-settings/).

The data will also be provided with a data dictionary. Other data that can be requested is described in the [Understanding your data](https://support.spotify.com/uk/article/understanding-your-data/) page.

The extended streaming history represents a log of every played track, podcast and audiobook since the account was created. This allows us to calculate stats spanning longer than each year's [Spotify Wrapped](https://support.spotify.com/uk/article/spotify-wrapped/). Some trends typically found in Spotify Wrapped will not be possible to calculate. However, it includes data other data such as skipped tracks.

<div class="alert alert-block alert-warning">
    <h3>Warning</h3>
    Before proceeding, please ensure you have a <code>.env</code> file with <code>DATA_FILEPATH</code> variable pointing to the directory containing the extended streaming json files.
</div>

## Caveats

- We only consider music and podcast usage (no audiobooks);
- This data does not include expected track length. Depending on the user, it would probably take thousands of requests to the Spotify API in order to have this data for every song. As such, we cannot make data checks based on this. Scenarios where having access to this info would be useful are described afterwards (namely for data cleanup).

## Table of contents

1. [Import and preprocess data](#Import-and-preprocess-data)
2. [Overall activity](#Overall-activity)
3. [Music to podcast ratio](#Music-to-podcast-ratio)
4. [Most listened tracks and artists](#Most-listened-tracks-and-artists)
5. [Song skip behaviour](#Song-skip-behaviour)
6. [Music genres](#Music-genres)

In [ ]:
from quoi.viz import (
    bold_text,
    build_subtitle,
    plot_area,
    plot_bar,
    plot_line,
    setup_chart_template
)
from quoi.stats import (
    fill_based_on,
    fill_cartesian_expansion,
    get_top
)
from quoi.time import duration_to_interval

from datetime import timedelta
import numpy as np
import polars as pl
import plotly.express as px
import plotly.graph_objects as go

import os
from dotenv import load_dotenv

load_dotenv()
setup_chart_template(from_notebook=True)

def ms_to_interval_str(x):
    interval = duration_to_interval(x, units='ms')
    
    day_str = '{}d '.format(interval['days']) if interval['days'] else ''
    hour_str = '{}h '.format(interval['hours'])
    minute_str = '{}m'.format(interval['minutes'])

    return day_str + hour_str + minute_str

## Dataset description

Each row in the dataset represents a streamed piece of media. We can group columns into 4 distinct categories:
1. User / stream info (includes timestamp, platform it was played, IP address, country, stream outcome, etc)
2. Music track info (name, album, artist, URI)
3. Podcast info (name, show, URI)
4. Audiobook info (title, URI, chapter info, etc)

Since a stream can only be made to a single media type, we can deduce which stream type it was based on the columns that were not empty. So for example, if a stream has music track information, then it was a song stream. No instances of rows with multiple stream type information were found in our data, but one row with no stream data was found (logging error?).

### About listening activity

This analysis will heavily rely on the `ms_played` parameter, as it is one of the two metrics we can make use of to measure usage on both music and podcasts (the other being row counts). There are two aspects to this column we would like to know so we can more confidently trust our results:

1. What is the margin for error in these readings?
   - What about for timestamps? Do they rely only on the device time?
3. Is it impacted by the usage of the track slider?

The second question poses a problem, as it makes it so that some "fully played" songs represent false positives. One example of this would be to skip the song by sliding the play cursor all the way to the end of the slider, which would likely end up as a `trackdone` log with a very low `ms_played`:

<center>
    <img alt="Music repeat" width="55%" src="media/slider_skip.png"/>
    <br>
    <figcaption><em><b>Figure 1:</b> Skipping second half of the song by interacting with the slider.</em></figcaption>
    <br>
</center>

The opposite example would be a user repeating parts of a song (or even the entire track):

<center>
    <img alt="Music repeat" width="55%" src="media/slider_repeat.png"/>
    <br>
    <figcaption><em><b>Figure 2:</b> Repeating second half of the song by interacting with the slider.</em></figcaption>
    <br>
</center>

The presence of instances of songs with very large amounts of played time makes us believe that `ms_played` represents the <span style="color: teal">total amount of time played without reaching any sort of signal marking the end of the play</span> (be it finish, skip, repeat, app exit, etc). As an example, with the data used for this analysis, the track instance with the highest `ms_played`, _White Ceiling_ by Parannoul, lasts 10 minutes, but was played for nearly 30.

This scenario could be handled by considering songs whose `ms_played` represents at least the vast majority of a song's duration (say 95% upwards) as "fully played" and keeping both metrics in mind for usage patterns. However, this would require having the expected track lengths for every entry. As it stands, for both of these cases, we cannot accurately discern which are logging / sync errors and which were due to slider usage, at least purely from the data perspective. The data itself looks correct, but we have no insight into slider usage.


### Activity metrics

Here is a brief comparison of our two metrics we will use to measure activity:

<div style="display: flex">
    <div style="flex: 1 1 0px">
        <h3>Play time</h3>
        <ul>
            <li>Calculated with sum of <code>ms_played</code> (in milliseconds);</li>
            <li>Can be used to measure overall activity more accurately (useful for podcast episodes heard in sessions);</li>
            <li><span style="color: tomato">Sensitive to slider usage;</span></li>
            <li><span style="color: tomato">Biased towards longer songs.</span></li>
        </ul>
    </div>
    <div style="flex: 1 1 0px">
        <h3>Play count</h3>
        <ul>
            <li>Calculated with row count;</li>
            <li>Not impacted by slider usage;</li>
            <li>Not impacted by track duration;</li>
            <li><span style="color: tomato">Requires <code>trackdone</code> outcome for accuracy and assumes stream is fully played.</span></li>
        </ul>
    </div>
</div>

## Import and preprocess data

This step will open the `Streaming_History_Audio_*` files found in the `DATA_FILEPATH` environment variable. If this variable does not exist, or the folder does not contain any relevant files, the runtime will raise an error. It also assumes the files are in JSON format.

In [ ]:
DATA_FILEPATH = os.getenv('DATA_FILEPATH')

if not DATA_FILEPATH:
    raise EnvironmentError("Please set 'DATA_FILEPATH' variable in .env to proceed.")

FILE_PATTERN = 'Streaming_History_Audio_'

schema = {
    'ts': pl.Datetime,
    'platform': pl.String,
    'ms_played': pl.Int64,
    'conn_country': pl.String,
    'ip_addr': pl.String,
    'master_metadata_track_name': pl.String,
    'master_metadata_album_artist_name': pl.String,
    'master_metadata_album_album_name': pl.String,
    'spotify_track_uri': pl.String,
    'episode_name': pl.String,
    'episode_show_name': pl.String,
    'spotify_episode_uri': pl.String,
    'audiobook_title': pl.String,
    'audiobook_uri': pl.String,
    'audiobook_chapter_uri': pl.String,
    'audiobook_chapter_title': pl.String,
    'reason_start': pl.String,
    'reason_end': pl.String,
    'shuffle': pl.Boolean,
    'skipped': pl.Boolean,
    'offline': pl.Boolean,
    'offline_timestamp': pl.String,
    'incognito_mode': pl.Boolean
}

_entries = []

for file in os.listdir(DATA_FILEPATH):
    if file.startswith(FILE_PATTERN):
        _entries.append(pl.read_json(os.path.join(DATA_FILEPATH, file), schema=schema))

if not _entries:
    raise EnvironmentError("No 'Streaming_History_Audio_*' files have been found.")

df_raw = pl.concat(_entries).sort('ts')
# We won't be using these
df_raw = df_raw.drop(['platform', 'conn_country', 'ip_addr', 'incognito_mode', 'offline_timestamp',
                      'spotify_track_uri', 'spotify_episode_uri',
                      'audiobook_title', 'audiobook_uri', 'audiobook_chapter_uri',
                      'audiobook_chapter_title'])

del _entries, DATA_FILEPATH

In [ ]:
# Music columns
MUSIC_COL = 'master_metadata_track_name'
ARTIST_COL = 'master_metadata_album_artist_name'
ALBUM_COL = 'master_metadata_album_album_name'

# Podcast columns
EPISODE_COL = 'episode_name'
PODCAST_COL = 'episode_show_name'

# Common filters
FLT_IS_MUSIC = ~pl.col(MUSIC_COL).is_null()
FLT_IS_PODCAST = ~pl.col(EPISODE_COL).is_null()
FLT_FULL_STREAM = pl.col('reason_end') == 'trackdone'
FLT_SKIP_STREAM = pl.col('reason_end').is_in(['backbtn', 'fwdbtn', 'endplay', 'unexpected-exit', 'unexpected-exit-while-paused'])

One problem we see is that it appears there are some chunks of duplicated data. Some tracks would appear played in full within seconds of each other, which if aggregated would inflate daily usage to impossible numbers (i.e.: above 24 hours). I do not know what causes this (maybe duplicated logs or wrong timestamps?), so I do not have a perfect method for handling it, but one possible approach would be to aggregate data into time intervals (say hourly intervals) and check for entries where the total duration played surpasses the interval itself:

In [ ]:
# We want the interval to be small enough to detect occurences, but also prevent ongoing tracks
# as much as possible
df_1h = df_raw.with_columns(pl.col('ts').dt.truncate('1h').alias('ts_1h'))
df_1h = df_1h.group_by(df_1h['ts_1h']).agg(pl.Expr.sum(pl.col('ms_played')))
df_1h = df_1h.with_columns((pl.col('ms_played') / 1000 / 60).alias('minutes_played'))

# We give a small tolerance here, to account for ongoing songs
# Tolerance is more or less the average song length
_tolerance = 4
df_1h.filter(pl.col('minutes_played') > (60 + _tolerance)).sort('minutes_played')

Some of these hour intervals register multiple, maybe even douzens of hours of playtime. Let's filter down to only songs played in full:<br>
<b style='color: tomato'>Note:</b> here we assume tracks were played in full if `reason_end = 'trackdone'`.

In [ ]:
df_1h_played = df_raw.with_columns(pl.col('ts').dt.truncate('1h').alias('ts_1h')).filter(FLT_FULL_STREAM)
df_1h_played = df_1h_played.group_by(df_1h_played['ts_1h']).agg(pl.Expr.sum(pl.col('ms_played')))
df_1h_played = df_1h_played.with_columns((pl.col('ms_played') / 1000 / 60).alias('minutes_played'))

_tolerance = 4
df_1h_played.filter(pl.col('minutes_played') > (60 + _tolerance)).sort('minutes_played')

This will reduce the numbers somewhat, both in duration, and number of rows.

Now we will aggregate data into 2 minute intervals and remove duplicated entries. While this may not solve the problem entirely, it may be as far as we can go on our end.

In [ ]:
duplicate_cols = ['_ts_group', MUSIC_COL, ARTIST_COL, ALBUM_COL, 'reason_end']

df_raw = df_raw.with_columns(pl.col('ts').dt.truncate('2m').alias('_ts_group'))
df_raw = df_raw.with_columns(df_raw[duplicate_cols].is_duplicated().alias('is_duplicate'))

We can see the impact by aggregating daily activity:

In [ ]:
df = df_raw.filter(~pl.col('is_duplicate')).drop(['_ts_group', 'is_duplicate'])

df_time = df_raw.group_by(df_raw['ts'].dt.date()).agg(pl.Expr.sum(pl.col('ms_played')))
df_time = fill_cartesian_expansion(df_time, time_c='ts')
df_time = df_time.with_columns((pl.col('ms_played') / 1000 / 60).alias('min_played'))

df_time_flt = df.group_by(df['ts'].dt.date()).agg(pl.Expr.sum(pl.col('ms_played')))
df_time_flt = fill_cartesian_expansion(df_time_flt, time_c='ts')
df_time_flt = df_time_flt.with_columns((pl.col('ms_played') / 1000 / 60).alias('min_played'))

fig = go.Figure()
fig.add_trace(go.Scatter(x=df_time['ts'], y=df_time['min_played'], name='All readings'))
fig.add_trace(go.Scatter(x=df_time_flt['ts'], y=df_time_flt['min_played'], name='No duplicates'))

subtitle = build_subtitle('Includes skipped tracks')

fig.update_layout(title_text=f'<b>Spotify audio usage over time</b>{subtitle}',
                  xaxis_title='<b>Time</b>',
                  yaxis_title='<b>Minutes played</b>')
fig.update_traces(hovertemplate='<br>'.join([
                      '%{x|%d %B %Y}',
                      '<b>%{yaxis.title.text}:</b> %{y}'
                      '<extra></extra>'
                  ]))
fig.show(renderer='notebook')

## Overall activity

How does streaming activity distribute throughout the week? We can aggregate data by `weekday - hour` groups and see if any patterns emerge. We will also show music and podcast only breakdowns:

In [ ]:
df_heatmap = df.group_by(
    pl.when(FLT_IS_MUSIC).then(pl.lit('Music'))
      .when(FLT_IS_PODCAST).then(pl.lit('Podcast'))
      .otherwise(pl.lit('Other')).alias('stream_type'),
    pl.col('ts').dt.weekday().alias('weekday_val'),
    pl.col('ts').dt.strftime('%A').alias('weekday'),
    pl.col('ts').dt.hour().alias('hour'),
).agg(pl.Expr.sum(pl.col('ms_played')))

df_heatmap = fill_cartesian_expansion(df_heatmap, entry_l=['stream_type', 'weekday_val', 'hour'])
df_heatmap = fill_based_on(df_heatmap, base='weekday_val', target='weekday')
df_heatmap = df_heatmap.with_columns(
    (pl.col('ms_played') / 1000 / 60).alias('min_played')
).sort('weekday_val', 'hour')

In [ ]:
fig = go.Figure()

# Music and podcasts
_curr_view = df_heatmap.group_by(['weekday_val', 'weekday', 'hour']).agg(
    pl.col('min_played').sum()
).sort('weekday_val', 'hour')
fig.add_trace(go.Heatmap(x=_curr_view['hour'],
                         y=_curr_view['weekday'],
                         z=_curr_view['min_played'], 
                         colorscale=[[0, '#FCF6F5'], [1, '#5C825B']],
                         colorbar=dict(title=dict(text='<b>Minutes</b>'))))

# Filtered views
for t in ('Music', 'Podcast'):
    _curr_view = df_heatmap.filter(pl.col('stream_type') == t).group_by(['weekday_val', 'weekday', 'hour']).agg(
        pl.col('min_played').sum()
    ).sort('weekday_val', 'hour')
    fig.add_trace(go.Heatmap(x=_curr_view['hour'],
                             y=_curr_view['weekday'],
                             z=_curr_view['min_played'], 
                             colorscale=[[0, '#FCF6F5'], [1, '#5C825B']],
                             colorbar=dict(title=dict(text='<b>Minutes</b>')),
                             visible=False))

_subtitle = build_subtitle('Total sum of stream minutes - Includes skipped entries')
_default_title = bold_text('Listening activity throughout the week') + _subtitle

fig.update_layout(title=_default_title,
                  height=700,
                  xaxis_title=bold_text('Hour'),
                  xaxis_ticksuffix=':00',
                  yaxis_title=bold_text('Weekday'),
                  updatemenus=[dict(
                      active=0,
                      buttons=list([
                          dict(label='Music and podcasts',
                               method='update',
                               args=[{'visible': [True, False, False]},
                                     {'title': {'text': _default_title}}]),
                          dict(label='Music track',
                               method='update',
                               args=[{'visible': [False, True, False]},
                                     {'title': {'text': bold_text('Music listening activity throughout the week') + _subtitle}}]),
                          dict(label='Podcast',
                               method='update',
                               args=[{'visible': [False, False, True]},
                                     {'title': {'text': bold_text('Podcast listening activity throughout the week') + _subtitle}}]),
                      ]),
                      direction='down',
                      showactive=True,
                      x=1,
                      y=1.1
                  )])

fig.update_traces(hovertemplate='<br>'.join([
                      '<b>Weekday:</b> %{x}',
                      '<b>Hour:</b> %{y}',
                      '<b>Minutes:</b> %{z}'
                      '<extra></extra>'
                  ]))
fig.show()

### Music to podcast ratio

In [ ]:
df_music_podcast = df.group_by(pl.col('ts').dt.date().alias('time')).agg(
    (pl.col('ms_played').filter(FLT_IS_PODCAST).sum() / 1000 / 60).alias('podcast_mins'),
    (pl.col('ms_played').filter(FLT_IS_MUSIC).sum() / 1000 / 60).alias('music_mins'),
)
df_music_podcast = fill_cartesian_expansion(df_music_podcast, time_c='time')

plot_line(df_music_podcast,
          x='time',
          y=['music_mins', 'podcast_mins'],
          title='Spotify audio usage broken down by music or podcast',
          subtitle='Includes skipped tracks',
          x_title='Time',
          y_title='Minutes played',
          legend_summary=True,
          legend_replace={'music_mins': 'Music', 'podcast_mins': 'Podcast'},
          hovertemplate='<br>'.join([
              '%{x|%d %B %Y}',
              '<b>%{yaxis.title.text}:</b> %{y}'
              '<extra></extra>'
          ]))

In [ ]:
df_music_podcast_monthly = df_music_podcast.group_by(pl.col('time').dt.truncate('1mo')).agg(
    pl.col('podcast_mins').sum(),
    pl.col('music_mins').sum(),
).sort(pl.col('time'))

df_music_podcast_monthly = df_music_podcast_monthly.unpivot(['music_mins', 'podcast_mins'],
                                                            index='time',
                                                            variable_name='stream_type',
                                                            value_name='stream_minutes')

plot_area(df_music_podcast_monthly,
          x='time',
          y='stream_minutes',
          breakdown='stream_type',
          title='Spotify audio usage broken down by music or podcast',
          subtitle='Includes skipped tracks',
          x_title='Time',
          y_title='Minutes played',
          legend_replace={'music_mins': 'Music', 'podcast_mins': 'Podcast'})

### Most listened tracks and artists

We will consider songs played in full only.

In [ ]:
# Top music tracks
MUSIC_AGG_COLS = (ARTIST_COL, ALBUM_COL, MUSIC_COL)

top_tracks_ms = get_top(df.filter(FLT_IS_MUSIC & FLT_FULL_STREAM),
                        entry_c=MUSIC_AGG_COLS,
                        value_c='ms_played').rename({'share': 'share_ts'})
top_tracks_count = get_top(df.filter(FLT_IS_MUSIC & FLT_FULL_STREAM),
                           entry_c=MUSIC_AGG_COLS,
                           # In theory, this should not make a difference
                           value_c='ms_played',
                           method=pl.Expr.count).rename({'ms_played': 'count', 'share': 'share_count'})

top_tracks = top_tracks_ms.join(top_tracks_count, on=MUSIC_AGG_COLS, how='full',
                                coalesce=True)
top_tracks = top_tracks.sort('share_ts', descending=True).with_columns(
    (pl.col('ms_played') / 1000 / 60).alias('min_played'),
    pl.col('share_ts').cum_sum().alias('share_ts_cumulative'),
    pl.col('share_count').cum_sum().alias('share_count_cumulative'),
).with_row_index()

# Top albums
top_albums = top_tracks.group_by([ALBUM_COL, ARTIST_COL]).agg(
    [pl.Expr.sum(pl.col(el)) for el in ['ms_played', 'min_played', 'share_ts', 'count', 'share_count']]
)
top_albums = top_albums.sort('share_ts', descending=True).with_columns(
    pl.col('share_ts').cum_sum().alias('share_ts_cumulative'),
    pl.col('share_count').cum_sum().alias('share_count_cumulative'),
).with_row_index()

# Top artists
top_artists = top_tracks.group_by(ARTIST_COL).agg(
    [pl.Expr.sum(pl.col(el)) for el in ['ms_played', 'min_played', 'share_ts', 'count', 'share_count']]
)
top_artists = top_artists.sort('share_ts', descending=True).with_columns(
    pl.col('share_ts').cum_sum().alias('share_ts_cumulative'),
    pl.col('share_count').cum_sum().alias('share_count_cumulative'),
).with_row_index()

In [ ]:
_top_n = 10

_title = 'Most played {}s by total play time'
_subtitle = 'Excluding skipped tracks'
_dropdown_views = []

fig = go.Figure()

_curr_view = top_tracks.head(_top_n)
fig.add_trace(go.Bar(x=_curr_view[MUSIC_COL].map_elements(lambda x: f'{x[:25]}...' if len(x) > 25 else x),
                     y=_curr_view['min_played'],
                     text=_curr_view['ms_played'].map_elements(lambda x: ms_to_interval_str(x)),
                     customdata=np.stack((_curr_view[MUSIC_COL],
                                          _curr_view[ALBUM_COL],
                                          _curr_view[ARTIST_COL],
                                          _curr_view['share_ts'],
                                          _curr_view['share_count']), axis=-1),
                     hovertemplate='<br>'.join([
                         '<b>%{xaxis.title.text}:</b> %{customdata[0]}',
                         '<b>Album</b>: %{customdata[1]}',
                         '<b>Artist</b>: %{customdata[2]}<br>',
                         '<b>Share of total time played:</b> %{customdata[3]:.3f}%',
                         '<b>%{yaxis.title.text}:</b> %{y}',
                         '<b>Share of total plays:</b> %{customdata[4]:.3f}%',
                         '<extra></extra>'
                     ])))
_dropdown_views.append('music track')

_curr_view = top_albums.head(_top_n)
fig.add_trace(go.Bar(x=_curr_view[ALBUM_COL].map_elements(lambda x: f'{x[:25]}...' if len(x) > 25 else x),
                     y=_curr_view['min_played'],
                     text=_curr_view['ms_played'].map_elements(lambda x: ms_to_interval_str(x)),
                     customdata=np.stack((_curr_view[ALBUM_COL],
                                          _curr_view[ARTIST_COL],
                                          _curr_view['share_ts'],
                                          _curr_view['share_count']), axis=-1),
                     hovertemplate='<br>'.join([
                         '<b>%{xaxis.title.text}:</b> %{customdata[0]}',
                         '<b>Artist</b>: %{customdata[1]}<br>',
                         '<b>Share of total time played:</b> %{customdata[2]:.3f}%',
                         '<b>%{yaxis.title.text}:</b> %{y}',
                         '<b>Share of total plays:</b> %{customdata[3]:.3f}%',
                         '<extra></extra>'
                     ]),
                     visible=False))
_dropdown_views.append('album')

_curr_view = top_artists.head(_top_n)
fig.add_trace(go.Bar(x=_curr_view[ARTIST_COL].map_elements(lambda x: f'{x[:25]}...' if len(x) > 25 else x),
                     y=_curr_view['min_played'],
                     text=_curr_view['ms_played'].map_elements(lambda x: ms_to_interval_str(x)),
                     customdata=np.stack((_curr_view[ARTIST_COL],
                                          _curr_view['share_ts'],
                                          _curr_view['share_count']), axis=-1),
                     hovertemplate='<br>'.join([
                         '<b>%{xaxis.title.text}:</b> %{customdata[0]}<br>',
                         '<b>Share of total time played:</b> %{customdata[1]:.3f}%',
                         '<b>%{yaxis.title.text}:</b> %{y}',
                         '<b>Share of total plays:</b> %{customdata[2]:.3f}%',
                         '<extra></extra>'
                     ]),
                     visible=False))
_dropdown_views.append('artist')

_dropdowns = [
    dict(
        active=0,
        buttons=list([
            dict(label=opt.capitalize(), method='update',
                 args=[{'visible': [el == opt for el in _dropdown_views]},
                       {'title': {'text': bold_text(_title.format(opt)) + build_subtitle(_subtitle)},
                        'xaxis': {'title': {'text': bold_text(opt.capitalize())},
                                  'tickangle': -25}}]) for opt in _dropdown_views
        ]),
        direction='down',
        showactive=True,
        x=1,
        y=1.15,
)]

fig.update_layout(title=bold_text(_title.format('music track')) + build_subtitle(_subtitle),
                  xaxis_title=bold_text('Music track'),
                  xaxis_tickangle=-25,
                  yaxis_title=bold_text('Time played (minutes)'),
                  updatemenus=_dropdowns,
                  height=700)
fig.update_traces(textposition='outside')
fig.show()

Let's take a look at how both metrics correlate, are there outliers?

In [ ]:
_title = 'Distribution of {}s by total plays and overall time'
_subtitle = 'Excluding skipped tracks'
_dropdown_views = []

fig = plot_line(top_tracks,
                x='min_played',
                y='count',
                title=_title.format('music track'),
                subtitle=_subtitle,
                x_title='Time played (minutes)',
                y_title='Total plays',
                return_fig=True)

fig.update_traces(customdata=np.stack((top_tracks[MUSIC_COL],
                                       top_tracks[ALBUM_COL],
                                       top_tracks[ARTIST_COL],
                                       top_tracks['share_ts'],
                                       top_tracks['share_count']), axis=-1),
                  mode='markers',
                  hovertemplate='<br>'.join([
                      '<b>Song</b>: %{customdata[0]}',
                      '<b>Album</b>: %{customdata[1]}',
                      '<b>Artist</b>: %{customdata[2]}',
                      '<b>%{xaxis.title.text}:</b> %{x}',
                      '<b>Share of total time played:</b> %{customdata[3]:.3f}%',
                      '<b>%{yaxis.title.text}:</b> %{y}',
                      '<b>Share of total plays:</b> %{customdata[4]:.3f}%',
                      '<extra></extra>'
                  ]))
_dropdown_views.append('music track')

# Add top albums view
fig.add_trace(go.Scatter(x=top_albums['min_played'], y=top_albums['count'],
                         customdata=np.stack((top_albums[ALBUM_COL],
                                              top_albums[ARTIST_COL],
                                              top_albums['share_ts'],
                                              top_albums['share_count']),
                                             axis=-1),
                         mode='markers',
                         hovertemplate='<br>'.join([
                             '<b>Album</b>: %{customdata[0]}',
                             '<b>Artist</b>: %{customdata[1]}',
                             '<b>%{xaxis.title.text}:</b> %{x}',
                             '<b>Share of total time played:</b> %{customdata[2]:.3f}%',
                             '<b>%{yaxis.title.text}:</b> %{y}',
                             '<b>Share of total plays:</b> %{customdata[3]:.3f}%',
                             '<extra></extra>'
                         ]),
                         visible=False))
_dropdown_views.append('album')

# Add top artists view
fig.add_trace(go.Scatter(x=top_artists['min_played'], y=top_artists['count'],
                         customdata=np.stack((top_artists[ARTIST_COL],
                                              top_artists['share_ts'],
                                              top_artists['share_count']),
                                             axis=-1),
                         mode='markers',
                         hovertemplate='<br>'.join([
                             '<b>Artist</b>: %{customdata[0]}',
                             '<b>%{xaxis.title.text}:</b> %{x}',
                             '<b>Share of total time played:</b> %{customdata[1]:.3f}%',
                             '<b>%{yaxis.title.text}:</b> %{y}',
                             '<b>Share of total plays:</b> %{customdata[2]:.3f}%',
                             '<extra></extra>'
                         ]),
                         visible=False))
_dropdown_views.append('artist')

_dropdowns = [
    dict(
        active=0,
        buttons=list([
            dict(label=opt.capitalize(), method='update',
                 args=[{'visible': [el == opt for el in _dropdown_views]},
                       {'title': {'text': bold_text(_title.format(opt)) + build_subtitle(_subtitle)}}]) for opt in _dropdown_views
        ]),
        direction='down',
        showactive=True,
        x=1,
        y=1.15,
)]

fig.update_layout(hoverlabel=dict(align='left'),
                  updatemenus=_dropdowns)
fig.show()

Strong outliers will deviate from the line established between total plays and played time towards the top or left border. This view focuses more on the overall relationship between the two metrics, but what about how much of total activity is accounted by the most played tracks? That is, how skewed is this distribution?

In [ ]:
_title = 'Cumulative distribution of {}s by total time and plays'
_subtitle = 'Excluding skipped tracks'
_dropdown_views = []

fig = plot_line(top_tracks,
                x='index',
                y=['share_ts_cumulative', 'share_count_cumulative'],
                title=_title.format('music track'),
                subtitle=_subtitle,
                x_title='Music track',
                y_title='Cumulative share',
                legend_replace={'share_ts_cumulative': 'Time played',
                                'share_count_cumulative': 'Total plays'},
                return_fig=True)
_dropdown_views.append('music track')


# Add top albums view
fig.add_trace(go.Scatter(x=top_albums['index'], y=top_albums['share_ts_cumulative'], name='Time played',
                         visible=False))
fig.add_trace(go.Scatter(x=top_albums['index'], y=top_albums['share_count_cumulative'], name='Total plays',
                         visible=False))
_dropdown_views.append('album')

# Add top artists view
fig.add_trace(go.Scatter(x=top_artists['index'], y=top_artists['share_ts_cumulative'], name='Time played',
                         visible=False))
fig.add_trace(go.Scatter(x=top_artists['index'], y=top_artists['share_count_cumulative'], name='Total plays',
                         visible=False))
_dropdown_views.append('artist')


_dropdowns = [
    dict(
        active=0,
        buttons=list([
            dict(label=opt.capitalize(), method='update',
                 args=[{'visible': [el == opt for dv in _dropdown_views for el in (dv, dv)]},
                       {'title': {'text': bold_text(_title.format(opt)) + build_subtitle(_subtitle)},
                        'xaxis': {'title': {'text': bold_text(opt.capitalize())}}}]) for opt in _dropdown_views
        ]),
        direction='down',
        showactive=True,
        x=1,
        y=1.25,
)]

fig.update_layout(yaxis_ticksuffix='%',
                  hoverlabel=dict(align='left'),
                  updatemenus=_dropdowns)

fig.show()

The steeper the initial curve, the more the most played entries make up total usage. Since we ordered tracks by activity, we expect the cumulative share curve to become more gradual along the x-axis.

<b style='color: tomato'>Note:</b> Tracks are actually ordered by time played. Depending on the data, there may be a discrepancy between both distributions, which may make it so that the <code>total plays</code> counterpart is not entirely ordered, but we expect this to have negligible impact.

### Discoverability vs replayability

Here we will bin music streams by whether or not they had already been played prior. Polars makes calculating this easy for us, as it has a method for calculating whether or not a given value is unique at a given time: `is_first_distinct`.

In [ ]:
df_discover = df.filter(FLT_IS_MUSIC & FLT_FULL_STREAM)
df_discover = df_discover.with_columns(
    pl.col('master_metadata_track_name').is_first_distinct().alias('music'),
    pl.col('master_metadata_album_artist_name').is_first_distinct().alias('artist'),
)

df_discover_monthly_counts = df_discover.unpivot(index='ts',
                                                 on=['music', 'artist'],
                                                 variable_name='entry_type',
                                                 value_name='is_distinct') \
                                        .group_by([pl.col('ts').dt.truncate('1mo'), 'entry_type', 'is_distinct']) \
                                        .len(name='count') \
                                        .sort('ts')

df_discover_monthly_counts = fill_cartesian_expansion(df_discover_monthly_counts,
                                                      time_c='ts',
                                                      entry_l=['entry_type', 'is_distinct'],
                                                      interval='1mo')

df_discover_monthly_counts = df_discover_monthly_counts.with_columns(
    ((pl.col('count') / pl.col('count').sum()).over([pl.col('ts'), pl.col('entry_type')]) * 100).alias("share")
)

In [ ]:
_title = '{} play counts by discovery or repeat stream'
_subtitle = 'Excluding skipped tracks'
_dropdown_views = []
_legend_replace = {True: 'Discovery', False: 'Repeat'}

fig = go.Figure()

for t in ('music', 'artist'):
    for status in (True, False):
        _curr_view = df_discover_monthly_counts.filter((pl.col('entry_type') == t)
                                                       & (pl.col('is_distinct') == status))
        _curr_name = _legend_replace.get(status)
        
        fig.add_trace(go.Scatter(x=_curr_view['ts'], y=_curr_view['count'],
                                 name=_curr_name,
                                 stackgroup='one',
                                 visible=(t == 'music')))
        _dropdown_views.append(f'{t} - play count')

        fig.add_trace(go.Scatter(x=_curr_view['ts'], y=_curr_view['share'],
                                 name=_curr_name,
                                 stackgroup='one',
                                 visible=False))
        _dropdown_views.append(f'{t} - share')

_dropdowns = [
    dict(
        active=0,
        buttons=list([
            dict(label=opt.capitalize(), method='update',
                 args=[{'visible': [el == opt for el in _dropdown_views]},
                       {'title': {'text': bold_text(_title.format(opt.split(' - ')[0].capitalize())) + build_subtitle(_subtitle)}}
                      ]) for opt in list(dict.fromkeys(_dropdown_views))
        ]),
        direction='down',
        showactive=True,
        x=1,
        y=1.25,
)]
# Percentage y-axis for ' - share' dropdown views
for i in range(len(_dropdowns[0]['buttons'])):
    if _dropdowns[0]['buttons'][i]['label'].endswith(' - share'):
        _dropdowns[0]['buttons'][i]['args'][1]['yaxis'] = {'ticksuffix': '%',
                                                           'range': (0, 100),
                                                           'title': {'text': bold_text('Share')}}
    # Undo y-axis changes
    else:
        _dropdowns[0]['buttons'][i]['args'][1]['yaxis'] = {'title': {'text': bold_text('Play counts')}}


fig.update_layout(title=bold_text(_title.format('Music')) + build_subtitle(_subtitle),
                  xaxis_title=bold_text('Time'),
                  yaxis_title=bold_text('Play counts'),
                  updatemenus=_dropdowns,
                  hovermode='x unified')
fig.show()

## Song skip behaviour

### Song skips by shuffle usage

In [ ]:
_sort_order = {'Skipped': 1, 'Played': 2, 'Other': 3}
df_skip = df.group_by(
    pl.col('shuffle'),
    pl.when(FLT_FULL_STREAM).then(pl.lit('Played'))
      .when(FLT_SKIP_STREAM).then(pl.lit('Skipped'))
      .otherwise(pl.lit('Other'))
      .alias('outcome')
).len(name='count').sort(['shuffle', pl.col('outcome').replace(_sort_order)],
                         descending=[True, False])

fig = plot_bar(df_skip,  # Trace order
               x='shuffle',
               y='count',
               breakdown='outcome',
               breakdown_stack=True,
               summary_sum=True,
               title='Distribution of song play outcome by shuffle usage',
               x_title='Shuffle',
               y_title='Total plays',
               legend_title='Outcome',
               legend_replace={True: 'Skipped', False: 'Played'},
               hovertemplate='%{y} (%{customdata:.2f}%)',
               return_fig=True)

fig.update_layout(hovermode='x unified')
fig.show()

### Most skipped tracks

<b style='color: tomato'>Note:</b> there are other outcomes besides being played in full or skipped (like interruptions caused by selecting other songs, or exiting the app). We will assume these occur rarely and that total occurences roughly equals times played + times skipped.

In [ ]:
df_skip_tracks = df.filter(FLT_IS_MUSIC).group_by([ARTIST_COL, ALBUM_COL, MUSIC_COL]).agg(
    pl.col('ms_played').count().alias('total_times'),
    pl.col('ms_played').filter(FLT_FULL_STREAM).count().alias('times_played'),
    pl.col('ms_played').filter(FLT_SKIP_STREAM).count().alias('times_skipped'),
)
df_skip_tracks = df_skip_tracks.with_columns(
    (pl.col('times_skipped') / (pl.col('times_played') + pl.col('times_skipped')) * 100).alias('skip_share')
)

In [ ]:
_subtitle = 'Songs only'
fig = plot_line(df_skip_tracks,
                x='total_times',
                y='skip_share',
                title='Distribution of skipped music tracks by total occurences',
                subtitle=_subtitle,
                x_title='Total song occurences',
                y_title='Skip occurence',
                return_fig=True)

fig.update_traces(customdata=np.stack((df_skip_tracks[MUSIC_COL],
                                       df_skip_tracks[ALBUM_COL],
                                       df_skip_tracks[ARTIST_COL],
                                       df_skip_tracks['times_played']), axis=-1),
                  mode='markers',
                  hovertemplate='<br>'.join([
                      '<b>Song</b>: %{customdata[0]}',
                      '<b>Album</b>: %{customdata[1]}',
                      '<b>Artist</b>: %{customdata[2]}<br>',
                      '<b>Times fully played:</b> %{customdata[3]}',
                      '<b>%{xaxis.title.text}:</b> %{x}',
                      '<b>%{yaxis.title.text}:</b> %{y:.2f}%',
                      '<extra></extra>'
                  ]))

fig.update_layout(yaxis_ticksuffix='%')
fig.show()

In [ ]:
_top_n = 25
fig = go.Figure()

_top_skipped_total = df_skip_tracks.sort('times_skipped', descending=True).head(_top_n).reverse()
fig.add_trace(go.Bar(x=_top_skipped_total['times_skipped'],
                     # Trim track names if too long
                     y=_top_skipped_total[MUSIC_COL].map_elements(lambda x: f'{x[:25]}...' if len(x) > 25 else x),
                     marker=dict(color=_top_skipped_total['skip_share'],
                                 cmin=0, cmax=100,
                                 colorscale=[[0, '#FCF6F5'], [1, '#D5834F']]),
                     customdata=np.stack((
                         _top_skipped_total[MUSIC_COL],
                         _top_skipped_total[ALBUM_COL],
                         _top_skipped_total[ARTIST_COL],
                         _top_skipped_total['total_times'],
                         _top_skipped_total['skip_share'],
                     ), axis=-1),
                     hovertemplate='<br>'.join([
                         '<b>Song</b>: %{customdata[0]}',
                         '<b>Album</b>: %{customdata[1]}',
                         '<b>Artist</b>: %{customdata[2]}',
                         '<b>Total song occurences:</b> %{customdata[3]}',
                         '<b>%{xaxis.title.text}:</b> %{x}',
                         '<b>Likelihood of skip:</b> %{customdata[4]:.2f}%'
                         '<extra></extra>'
                     ]),
                     orientation='h'))

fig.update_layout(title=bold_text('Most skipped tracks'),
                  xaxis_title=bold_text('Total times skipped'))
fig.update_traces(text=_top_skipped_total['times_skipped'],
                  textposition='outside',
                  textfont=dict(size=15))
fig.show()

### Distribution of track skips by time played

How long does it take for us to skip a song? Let's calculate the distribution of seconds played before the song was skipped.

In [ ]:
df_skip_duration = df.filter(FLT_IS_MUSIC & FLT_SKIP_STREAM) \
                     .group_by('ms_played') \
                     .len(name='count') \
                     .sort('ms_played')

df_skip_duration = df_skip_duration.with_columns(
    (pl.col('ms_played') / 1000).alias('s_played')
)

In [ ]:
fig = px.histogram(df_skip_duration, x='s_played', y='count')

fig.update_layout(title=bold_text('Distribution of track skips by time played prior to skip'),
                  xaxis_title=bold_text('Seconds played'),
                  yaxis_title=bold_text('Skip count') + '<br>Logarithmic',
                  yaxis_type='log')
fig.show()

### Biggest skip streaks

This is an island grouping problem. We have chunks within data that reflect consistent song skips, and we want to calculate the biggest ones. Since this is a 1-dimensional counterpart, we do not need to rely on BFS, but rather on conditional sums.

Our condition is this: a row is considered if it represents a skipped song and not much time has passed since the previous row (let's say, about 10 minutes).

<b style='color: tomato'>Note:</b> this is done by filtering out non-music stream rows at the start, so in theory we are disregarding cases where the streak would end by switching to podcasts.

In [ ]:
df_skip_streak_raw = df.filter(FLT_IS_MUSIC)[['ts', 'reason_end']].sort('ts')

# Song was skipped, used to start a streak
start_streak_flag = FLT_SKIP_STREAM

# Song was skipped and not much time passed since last record, used to maintain a streak
streak_flag = (((pl.col('ts') - pl.col('ts').shift()).fill_null(timedelta(seconds=0)) < timedelta(seconds=60*10))
               & FLT_SKIP_STREAM)

df_skip_streak_raw = df_skip_streak_raw.with_columns(
    # Notice the "over" expression effectively acts as a reset because it calculates a sort of unique Id
    # every time a song is not skipped
    ((~streak_flag).cum_sum()).alias('streak_id'),
    pl.when(streak_flag).then(streak_flag.cast(pl.Int32).cum_sum().over((~streak_flag).cum_sum()))
    # Streak resets - start at either 0 for non-skip or 1 for skip
    .otherwise(start_streak_flag.cast(pl.Int32))
    .alias('skip_streak')
)

# Aggregate into streak info per row
# Filter out small streaks, as there could be many of them
df_skip_streak_summary = df_skip_streak_raw.group_by('streak_id').agg(
    pl.col('ts').min().alias('start_ts'),
    pl.col('ts').max().alias('end_ts'),
    pl.col('skip_streak').max().alias('size')
).filter(pl.col('size') > 5)

In [ ]:
_top_n = 25
fig = go.Figure()

_top_skip_streaks = df_skip_streak_summary.sort('size', descending=True).head(_top_n).reverse().with_row_index()

fig.add_trace(go.Bar(x=_top_skip_streaks['size'], y=_top_skip_streaks['index'],
                     customdata=np.stack((_top_skip_streaks['start_ts'],
                                          _top_skip_streaks['end_ts']), axis=-1),
                     orientation='h',
                     hovertemplate='<br>'.join([
                         '<b>Streak:</b> %{x}',
                         '<b>Start time:</b> %{customdata[0]|%Y-%m-%d %H:%M:%S}',
                         '<b>End time:</b> %{customdata[1]|%Y-%m-%d %H:%M:%S}'
                         '<extra></extra>'
                     ])))

fig.update_layout(title=bold_text('Biggest song skip streaks'),
                  xaxis_title=bold_text('Total consecutive skips'),
                  yaxis_showticklabels=False,
                  yaxis_ticks='')

fig.update_traces(text=_top_skip_streaks['size'],
                  textposition='outside',
                  textfont=dict(size=15))

fig.show()

Results may include long streaks lasting mere seconds. Cases such as these re-raise the concern of how accurate the timestamps are.

## Future work

There are two extra dimensions that could add more interesting stats: genres and audio features.

With the Spotify API, genres are not set at the track level, but rather at the [artist level](https://developer.spotify.com/documentation/web-api/reference/get-an-artist). This makes it so that genre distributions could end up overgeneralized, depending on the artist.

With [audio features](https://developer.spotify.com/documentation/web-api/reference/get-audio-features), we could expand the analysis towards the type of music the user listens to. Fields like `tempo`, `energy` and `valence` could be used to know if there is a preference for more upbeat music, `acousticness` could be used to breakdown music into synthetic / electric vs acoustic and traditional, `instrumentalness` for instrumental music, as well as other distribution types.

Unfortunately, at the time of writing (July 2026), <span style='color: tomato'>both of these methods are deprecated</span>. For genres, a possible alternative would be to rely on Last.fm.

Other trends to consider include:
- Do we listen to popular bands or more underground artists? We could look into the distribution of (top) artists by audience size (`followers` field, <span style='color: tomato'>also deprecated</span>);
- How many of the artist's songs do we frequently listen to? Share of all artist's music that is frequently streamed; 
    - Same thing for albums;
- What music eras do we listen to the most? Distribution of listened songs by release year;
- Distribution of listened songs by album vs single.